# Kaggriculture | Adaptive Farm Intelligence — Fast Routes

A compact Kaggriculture submission package.


## Method

Exact, self-contained packaging of the public V48 Fast Routes agent. No route, timing, or market logic has been modified. The notebook writes `submission.tar.gz`.


In [ ]:
# Exact V48 Fast Routes competition package.
import base64
import gzip
import hashlib
import io
import tarfile
from pathlib import Path

AGENT_B64 = (
    '...'
)
EXPECTED_AGENT_SHA256 = 'dadee25a9840313218384208c53b2c4752f82c3209cc654632e0b96c65e2664a'
EXPECTED_ARCHIVE_SHA256 = "f2973a7c7636e1f95cea74c8c2cb7bbfec20f4082a9da079aeb8ba74fbe8b4fe"

agent = base64.b64decode(AGENT_B64)
assert hashlib.sha256(agent).hexdigest() == EXPECTED_AGENT_SHA256
compile(agent, "main.py", "exec")
stream = io.BytesIO()
with gzip.GzipFile(fileobj=stream, mode="wb", mtime=0) as zipped:
    with tarfile.open(fileobj=zipped, mode="w") as archive:
        info = tarfile.TarInfo("main.py")
        info.size = len(agent)
        info.mode = 0o644
        info.mtime = 0
        archive.addfile(info, io.BytesIO(agent))
payload = stream.getvalue()
assert hashlib.sha256(payload).hexdigest() == EXPECTED_ARCHIVE_SHA256
Path("submission.tar.gz").write_bytes(payload)
print("submission.tar.gz ready")
print("agent sha256:", EXPECTED_AGENT_SHA256)
print("archive sha256:", EXPECTED_ARCHIVE_SHA256)
